In [175]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

In [176]:
df = pd.read_csv('qoute_dataset.csv')

In [177]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [178]:
df.shape

(3038, 2)

In [179]:
quotes = df['quote']
quotes

0       “The world as we have created it is a process ...
1       “It is our choices, Harry, that show what we t...
2       “There are only two ways to live your life. On...
3       “The person, be it gentleman or lady, who has ...
4       “Imperfection is beauty, madness is genius and...
                              ...                        
3033        The past beats inside me like a second heart.
3034    Damn, Claire. Warn a guy before you do a face-...
3035    Can you be a girl for a few seconds?""I'm alwa...
3036    That's what fiction is for. It's for getting a...
3037    If we have no peace, it is because we have for...
Name: quote, Length: 3038, dtype: str

In [180]:
quotes = quotes.str.lower()
quotes 

0       “the world as we have created it is a process ...
1       “it is our choices, harry, that show what we t...
2       “there are only two ways to live your life. on...
3       “the person, be it gentleman or lady, who has ...
4       “imperfection is beauty, madness is genius and...
                              ...                        
3033        the past beats inside me like a second heart.
3034    damn, claire. warn a guy before you do a face-...
3035    can you be a girl for a few seconds?""i'm alwa...
3036    that's what fiction is for. it's for getting a...
3037    if we have no peace, it is because we have for...
Name: quote, Length: 3038, dtype: str

In [181]:
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))
quotes 

0       “the world as we have created it is a process ...
1       “it is our choices harry that show what we tru...
2       “there are only two ways to live your life one...
3       “the person be it gentleman or lady who has no...
4       “imperfection is beauty madness is genius and ...
                              ...                        
3033         the past beats inside me like a second heart
3034    damn claire warn a guy before you do a facepla...
3035    can you be a girl for a few secondsim always a...
3036    thats what fiction is for its for getting at t...
3037    if we have no peace it is because we have forg...
Name: quote, Length: 3038, dtype: str

In [182]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence  import pad_sequences

In [183]:
vocab_size = 10000
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)
sequences = tokenizer.texts_to_sequences(quotes)

In [184]:
word_index = tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [185]:

X = []
y = []

for seq in sequences:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)

In [186]:
len(X)

85271

In [187]:
len(y)

85271

In [188]:
maxlen = max(len(s) for s in sequences)
X_padded = pad_sequences(sequences, maxlen=maxlen, padding='pre')
X_padded

array([[   0,    0,    0, ...,  752,   70, 2461],
       [   0,    0,    0, ...,   54,   70, 3676],
       [   0,    0,    0, ...,    7,    5, 3677],
       ...,
       [   0,    0,    0, ...,   26, 2411, 8978],
       [   0,    0,    0, ...,   17,    1,  174],
       [   0,    0,    0, ...,    3,  169,  101]],
      shape=(3038, 746), dtype=int32)

In [189]:
y = np.array(y)

In [190]:
from tensorflow.keras.utils import to_categorical
one_hot_encoded_y = to_categorical(y, num_classes=vocab_size)

In [191]:
one_hot_encoded_y.shape

(85271, 10000)

In [192]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense

In [193]:
embedding_dim = 50
rnn_units = 128

In [194]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=maxlen,name='embed')
)

rnn_model.add(
    SimpleRNN(units=rnn_units, name='simple_rnn')
)

rnn_model.add(
    Dense(
        vocab_size, activation='softmax'
    )
)

In [195]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [196]:
rnn_model.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embed (Embedding)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [197]:
lstm_model = Sequential()

lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=maxlen,name='embed')
)

lstm_model.add(
    LSTM(units=rnn_units, name='lstm')
)

lstm_model.add(
    Dense(
        vocab_size, activation='softmax'
    )
)

In [198]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [199]:
lstm_model.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embed (Embedding)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [200]:
epochs=10
batch_size=128

In [201]:
rnn_model.fit(
    X_padded, 
    one_hot_encoded_y, 
    epochs=epochs, 
    batch_size=batch_size, 
    verbose=1,
    validation_split=0.1
    )

Epoch 1/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 9s 359ms/step - accuracy: 0.0044 - loss: 8.9503 - val_accuracy: 0.0033 - val_loss: 8.0520
Epoch 2/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 361ms/step - accuracy: 0.0256 - loss: 6.6570 - val_accuracy: 0.0230 - val_loss: 6.5150
Epoch 3/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 341ms/step - accuracy: 0.0333 - loss: 5.8647 - val_accuracy: 0.0822 - val_loss: 6.6525
Epoch 4/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 377ms/step - accuracy: 0.0384 - loss: 5.8155 - val_accuracy: 0.0822 - val_loss: 6.7200
Epoch 5/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 378ms/step - accuracy: 0.0315 - loss: 5.7999 - val_accuracy: 0.0822 - val_loss: 6.7652
Epoch 6/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 341ms/step - accuracy: 0.0274 - loss: 5.7947 - val_accuracy: 0.0822 - val_loss: 6.7891
Epoch 7/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 366ms/step - accuracy: 0.0384 - loss: 5.7879 - val_accuracy: 0.0822 - val_loss: 6.8368
Epoch 8/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 355ms/step - accuracy: 0.0355 - loss: 5.7875 - val_accuracy: 0.

In [202]:
epochs=100
batch_size=128

In [203]:
lstm_model.fit(
    X_padded, 
    one_hot_encoded_y, 
    epochs=epochs, 
    batch_size=batch_size, 
    verbose=1,
    validation_split=0.1
)

Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 23s 998ms/step - accuracy: 0.0318 - loss: 8.9059 - val_accuracy: 0.0822 - val_loss: 7.7001
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 20s 914ms/step - accuracy: 0.0384 - loss: 6.4937 - val_accuracy: 0.0822 - val_loss: 6.5200
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 18s 810ms/step - accuracy: 0.0384 - loss: 5.9179 - val_accuracy: 0.0822 - val_loss: 6.6460
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 17s 763ms/step - accuracy: 0.0384 - loss: 5.8421 - val_accuracy: 0.0822 - val_loss: 6.6996
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 16s 736ms/step - accuracy: 0.0278 - loss: 5.8128 - val_accuracy: 0.0822 - val_loss: 6.7356
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 16s 742ms/step - accuracy: 0.0384 - loss: 5.8035 - val_accuracy: 0.0822 - val_loss: 6.7808
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 17s 773ms/step - accuracy: 0.0355 - loss: 5.8007 - val_accuracy: 0.0822 - val_loss: 6.8209
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 17s 788ms/step - accuracy: 0.0384 - loss: 5.7919 - 

In [206]:
lstm_model.save('lstm_model.h5')

In [207]:
from tensorflow.keras.models import load_model

lstm_model = load_model("lstm_model.h5")

In [209]:

import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokenizer, f)

In [211]:

with open("max_len.pkl", "wb") as f:
  pickle.dump(maxlen, f)